# Practical Session 5 Bonus: Classes

In Python, **classes** allow us to create our own data types that combine data (attributes) and behavior (methods). This is the foundation of **object-oriented programming (OOP)**.

##### Why Use Classes?
- Group related data and functions together
- Model real-world or abstract systems cleanly
- Create reusable, extensible components

A simple class contains:

- `__init__` — the **constructor** that runs when you create an object
- `self` — a reference to the instance (object)
- **Attributes** — variables tied to the object
- **Methods** — functions that belong to the class

##### Example: A Simple `GasCloud` Class

We'll model a gas cloud in space with a temperature and mass, and give it a method to calculate its thermal energy.

In [1]:
class GasCloud:
    def __init__(self, mass, temperature):
        self.mass = mass                # in kilograms
        self.temperature = temperature  # in Kelvin

    def thermal_energy(self):
        k_B = 1.38e-23  # Boltzmann constant
        return 1.5 * k_B * self.temperature * self.mass

    def summary(self):
        return f"Gas cloud: {self.mass:.2e} kg, {self.temperature} K"

In [2]:
cloud = GasCloud(mass=1e30, temperature=5000)

print(cloud.summary())

E = cloud.thermal_energy()
print(f"Thermal energy: {E:.2e} J")

Gas cloud: 1.00e+30 kg, 5000 K
Thermal energy: 1.04e+11 J


### Task: Orbit Simulator
Here we will take the code from the practical session 5 and alter it from a function format into a class format. 

#### 1. Create a class called OrbitSimulator with an __init__ method that stores:
- the orbital radius `r`
- the mass of the central body `M`
- the number of time steps `steps` (default `100`)
- the gravitational constant `G` (default `6.67430e-11`)

```python
class OrbitSimulator:
    def __init__(self, r, M, steps=100, G=6.67430e-11):
        # your code here
```
#### 2. Add a method (class function) which `orbital_speed` that returns the orbital speed, and test it with values for the Earth and Sun
```python
earth_orbit = OrbitSimulator(r=1.496e11, M=1.989e30)
print(f"Speed: {earth_orbit.orbital_speed():.2e} m/s")
```
#### 3. Add a method (class function) which `orbital_period` that returns the orbital period, and test it with values for the Earth and Sun
```python
print(f"Period: {earth_orbit.orbital_period() / (60*60*24):.2f} days")
```

#### 4. Add a method `simulate_positions()` that returns a list of `(x, y)` positions of the orbiting body over one full orbit

In [3]:
# Math helper functions outside the class

def sqrt(x):
    guess = x / 2.0
    for _ in range(10):
        guess = (guess + x / guess) / 2
    return guess

def cos(theta):
    result = 1
    term = 1
    sign = -1
    for i in range(1, 6):
        term = term * theta * theta / ((2*i-1)*2*i)
        result += sign * term
        sign *= -1
    return result

def sin(theta):
    result = 0
    term = theta
    sign = 1
    for i in range(1, 6):
        result += sign * term
        term = term * theta * theta / ((2*i)*(2*i+1))
        sign *= -1
    return result

def pi():
    pi_val = 0
    for i in range(10000):
        pi_val += ((-1)**i) / (2*i + 1)
    return 4 * pi_val


class OrbitSimulator:
    def __init__(self, r, M, steps=100, G=6.67430e-11):
        self.r = r
        self.M = M
        self.steps = steps
        self.G = G
    
    def orbital_speed(self):
        return sqrt(self.G * self.M / self.r)
    
    def orbital_period(self):
        pi_val = pi()
        v = self.orbital_speed()
        return 2 * pi_val * self.r / v
    
    def simulate_positions(self):
        pi_val = pi()
        v = self.orbital_speed()
        T = self.orbital_period()
        omega = 2 * pi_val / T
        dt = T / self.steps
        positions = []
        
        for step in range(self.steps):
            t = step * dt
            angle = omega * t
            x = self.r * cos(angle)
            y = self.r * sin(angle)
            positions.append((x, y))
        
        return positions


# Test with Earth-Sun parameters
earth_orbit = OrbitSimulator(r=1.496e11, M=1.989e30)
period = earth_orbit.orbital_period()
positions = earth_orbit.simulate_positions()

print(f"Orbital period (seconds): {period:.2e}")
print(f"Orbital period (days): {period / (3600*24):.2f}")
print("First 5 positions:")
for pos in positions[:5]:
    print(pos)


Orbital period (seconds): 2.17e+06
Orbital period (days): 25.07
First 5 positions:
(149600000000.0, 0.0)
(149304817359.4942, 9393163111.970072)
(148420434314.86533, 18749258059.63409)
(146950340899.8343, 28031362960.485744)
(144900338532.26263, 37202847918.34827)


### Task: Gas Cloud Simulator

Let's return to the gas cloud class given in the example, and make some improvements.

#### 1. Add a method called `is_collapsing`. 
A gas cloud begins collapsing when gravity dominates over thermal motion. Use the following simplified comparison:

Escape velocity:
$$
\nu_e = \sqrt{\frac{2GM}{R}}
$$

Thermal particle speed:

$$
v_t=\sqrt{\frac{3k_BT}{m_p}}
$$

Constants:
```
G = 6.67e-11 
k_B = 1.38e-23 
m_p = 1.67e-27
```
This should return `True` if the escape velocity is greater than the thermal particle speed. You will need to add a radius when initialising this class.

#### 2. Cloud Merging System

Implement a method called `merge(other)` that combines two gas clouds into a completely new `GasCloud`.

##### Rules:

Mass - 
Combines linerally
$$
m_{\rm new} = m_1 +m_2
$$

Temperature -
Use a mass-weighted average
$$
T_{\rm new}=\frac{m_1T_1+m_2T_2}{m_1+m_2}
$$

Radius -
Assume volume is conserved
$$
R_{\rm new}=\left(R_1^3+R_2^3\right)^{1/3}
$$

This method must return a new object, leaving the original gas clouds unchanged. The returned object must also be a `GasCloud`.

Test your code with
```
c1 = GasCloud(5e30, 100, 4e11) 
c2 = GasCloud(2e30, 400, 2e11) 
c3 = c1.merge(c2) 

print(c3.summary())
```

#### 3. Collapse time

Implement `simulate(years)` where each simulated year:

- decreases temperature by 0.5%
- decreases radius by 0.1%
- recomputes whether collapse has started

At the end, return the number of years required before collapse first occurs.

If collapse never occurs, return None. Test how long collapse takes with the following
```
c = GasCloud(
    mass=2e29,       
    temperature=5000,
    radius=1e12 
)
```

In [6]:
class GasCloud:
    def __init__(self, mass, temperature, radius):
        self.mass = mass
        self.temperature = temperature
        self.radius = radius

    def thermal_energy(self):
        k_B = 1.38e-23
        return 1.5 * k_B * self.temperature * self.mass

    def summary(self):
        return (
            f"Gas cloud: "
            f"{self.mass:.2e} kg, "
            f"{self.temperature} K, "
            f"radius={self.radius:.2e} m"
        )

    # -----------------------------
    # Task 1 Solution
    # -----------------------------
    def is_collapsing(self):

        G = 6.67e-11
        k_B = 1.38e-23
        m_p = 1.67e-27

        # Escape velocity
        v_escape = ((2 * G * self.mass) / self.radius) ** 0.5

        # Thermal particle speed
        v_thermal = ((3 * k_B * self.temperature) / m_p) ** 0.5

        return v_escape > v_thermal

    # -----------------------------
    # Task 2 Solution
    # -----------------------------
    def merge(self, other):

        # New mass
        new_mass = self.mass + other.mass

        # Mass-weighted temperature
        new_temperature = (
            (self.mass * self.temperature) +
            (other.mass * other.temperature)
        ) / new_mass

        # Combined radius from combined volumes
        new_radius = (
            (self.radius ** 3) +
            (other.radius ** 3)
        ) ** (1 / 3)

        return GasCloud(
            new_mass,
            new_temperature,
            new_radius
        )

    # -----------------------------
    # Task 3 Solution
    # -----------------------------
    def simulate(self, years):

        for year in range(1, years + 1):

            # Cooling
            self.temperature *= 0.995

            # Contraction
            self.radius *= 0.999

            # Check for collapse
            if self.is_collapsing():
                return year

        return None


# ---------------------------------
# Example Usage
# ---------------------------------

c1 = GasCloud(5e30, 100, 4e11)
c2 = GasCloud(2e30, 400, 2e11)

print(c1.summary())
print(c2.summary())

print("\nCollapsing?")
print(c1.is_collapsing())

print("\nMerged cloud:")
c3 = c1.merge(c2)
print(c3.summary())

c = GasCloud(
    mass=2e29,        
    temperature=5000, 
    radius=1e12     
)

years = c.simulate(20000)

print(years)


Gas cloud: 5.00e+30 kg, 100 K, radius=4.00e+11 m
Gas cloud: 2.00e+30 kg, 400 K, radius=2.00e+11 m

Collapsing?
True

Merged cloud:
Gas cloud: 7.00e+30 kg, 185.71428571428572 K, radius=4.16e+11 m
256
